# Multiple MCP Servers with LangChain Agent 

This notebook runs **three independent MCP servers directly from notebook cells**:
- Country MCP — REST Countries API
- Weather MCP — Open-Meteo API
- Flight MCP — demonstration dataset (not live flight data)

**No `.py` files, project folders, or stdio subprocesses are created.**

We use Streamable HTTP on localhost because the original Windows/Jupyter `stdio` subprocess setup raised `UnsupportedOperation: fileno`.
Run the cells from top to bottom. If you change package versions, restart the kernel and run from the first cell.

## 1. Install dependencies

Run this cell once. If packages are updated, restart the Jupyter kernel before continuing.

In [1]:
%pip install -U langchain langgraph langchain-groq langchain-mcp-adapters "mcp>=1.9" requests

  Using cached mcp-2.3.0-py3-none-any.whl.metadata (6.6 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and API key

Enter your Groq API key when prompted. It is not printed or stored in this notebook.

In [4]:
import os
import sys
import time
import socket
import threading
from getpass import getpass

import requests
from mcp.server.fastmcp import FastMCP
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_groq import ChatGroq
from langchain.agents import create_agent

# if not os.environ.get("GROQ_API_KEY","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"):
#     os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ").strip()

# if not os.environ.get("GROQ_API_KEY"):
#     raise ValueError("GROQ_API_KEY is empty. Run this cell again and enter a valid key.")

print("Imports successful.")
print("Groq API key available:", bool(os.environ.get("GROQ_API_KEY")))
print("Python:", sys.version)

Imports successful.
Groq API key available: False
Python: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]


## 3. Define the Country MCP server

This server is defined in this cell and remains in notebook memory. It does not write a Python file.

In [5]:
country_mcp = FastMCP("Country MCP Server", host="127.0.0.1", port=8001)
COUNTRY_API = "https://restcountries.com/v3.1"

@country_mcp.tool()
def get_country_details(country: str) -> dict:
    """Get basic details such as capital, region, population, currencies and languages for a country."""
    response = requests.get(
        f"{COUNTRY_API}/name/{country}",
        params={"fields": "name,capital,region,subregion,population,currencies,languages"},
        timeout=20,
    )
    response.raise_for_status()
    results = response.json()
    if not results:
        raise ValueError(f"No country found for: {country}")
    item = results[0]
    return {
        "country": item.get("name", {}).get("common"),
        "capital": item.get("capital", []),
        "region": item.get("region"),
        "subregion": item.get("subregion"),
        "population": item.get("population"),
        "currencies": item.get("currencies", {}),
        "languages": item.get("languages", {}),
    }

@country_mcp.tool()
def get_country_capital(country: str) -> dict:
    """Find the capital city of a country by its name."""
    response = requests.get(
        f"{COUNTRY_API}/name/{country}",
        params={"fields": "name,capital"},
        timeout=20,
    )
    response.raise_for_status()
    results = response.json()
    if not results:
        raise ValueError(f"No country found for: {country}")
    item = results[0]
    return {
        "country": item.get("name", {}).get("common"),
        "capital": item.get("capital", []),
    }

print("Country MCP tools defined.")

Country MCP tools defined.


## 4. Define the Weather MCP server

Uses Open-Meteo geocoding and forecast APIs. No weather API key is required.

In [6]:
weather_mcp = FastMCP("Weather MCP Server", host="127.0.0.1", port=8002)
GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

def get_city_coordinates(city: str) -> dict:
    response = requests.get(
        GEOCODING_URL,
        params={"name": city, "count": 1, "language": "en", "format": "json"},
        timeout=20,
    )
    response.raise_for_status()
    results = response.json().get("results", [])
    if not results:
        raise ValueError(f"Could not find city: {city}")
    place = results[0]
    return {
        "name": place["name"],
        "country": place.get("country"),
        "latitude": place["latitude"],
        "longitude": place["longitude"],
    }

@weather_mcp.tool()
def get_current_weather(city: str) -> dict:
    """Get current weather conditions for a city."""
    place = get_city_coordinates(city)
    response = requests.get(
        FORECAST_URL,
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,weather_code,wind_speed_10m",
            "timezone": "auto",
        },
        timeout=20,
    )
    response.raise_for_status()
    return {"location": place, "current": response.json().get("current", {})}

@weather_mcp.tool()
def get_weather_forecast(city: str) -> dict:
    """Get a seven-day daily weather forecast for a city."""
    place = get_city_coordinates(city)
    response = requests.get(
        FORECAST_URL,
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max",
            "forecast_days": 7,
            "timezone": "auto",
        },
        timeout=20,
    )
    response.raise_for_status()
    return {"location": place, "daily": response.json().get("daily", {})}

print("Weather MCP tools defined.")

Weather MCP tools defined.


## 5. Define the Flight MCP server

This server uses sample data only. It does not provide live airline inventory, prices, bookings, or live flight status.

In [7]:
flight_mcp = FastMCP("Flight MCP Server", host="127.0.0.1", port=8003)

SAMPLE_FLIGHTS = [
    {"flight_number": "DEMO101", "airline": "Demo Air", "origin": "HYD",
     "destination": "DEL", "departure": "09:00", "arrival": "11:15",
     "price_inr": 5200, "status": "On time"},
    {"flight_number": "DEMO202", "airline": "Demo Air", "origin": "HYD",
     "destination": "LHR", "departure": "02:30", "arrival": "12:00",
     "price_inr": 42000, "status": "Scheduled"},
    {"flight_number": "DEMO303", "airline": "Sample Airways", "origin": "DEL",
     "destination": "LHR", "departure": "06:45", "arrival": "13:10",
     "price_inr": 38500, "status": "Scheduled"},
]

@flight_mcp.tool()
def search_flights(origin: str, destination: str) -> list:
    """Search the demonstration flight dataset using airport codes such as HYD and DEL."""
    origin = origin.strip().upper()
    destination = destination.strip().upper()
    return [
        flight for flight in SAMPLE_FLIGHTS
        if flight["origin"] == origin and flight["destination"] == destination
    ]

@flight_mcp.tool()
def get_flight_status(flight_number: str) -> dict:
    """Get a sample flight status. This is demonstration data, not a live status feed."""
    for flight in SAMPLE_FLIGHTS:
        if flight["flight_number"].upper() == flight_number.strip().upper():
            return {
                "flight_number": flight["flight_number"],
                "status": flight["status"],
                "note": "Demonstration data only; not a live airline status feed.",
            }
    return {
        "flight_number": flight_number,
        "status": "Not found in demo dataset",
        "note": "Demonstration data only; not a live airline status feed.",
    }

print("Flight MCP tools defined.")

Flight MCP tools defined.


## 6. Start all three MCP servers from this notebook

Each server runs in a background thread and listens on a different localhost port. No child Python scripts are launched.

If you rerun this cell, it checks whether each port is already listening before starting another thread.

In [8]:
def port_is_open(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.5)
        return sock.connect_ex((host, port)) == 0

server_specs = [
    ("Country", country_mcp, 8001),
    ("Weather", weather_mcp, 8002),
    ("Flight", flight_mcp, 8003),
]

server_threads = globals().get("server_threads", [])

for server_name, server, port in server_specs:
    if port_is_open("127.0.0.1", port):
        print(f"{server_name} server already listening on port {port}.")
        continue

    thread = threading.Thread(
        target=server.run,
        kwargs={"transport": "streamable-http"},
        name=f"{server_name}-MCP",
        daemon=True,
    )
    thread.start()
    server_threads.append(thread)

    deadline = time.time() + 20
    while time.time() < deadline and not port_is_open("127.0.0.1", port):
        time.sleep(0.25)

    if not port_is_open("127.0.0.1", port):
        raise RuntimeError(
            f"{server_name} MCP server did not start on port {port}. "
            "Check the error printed above. If the port is occupied, restart the kernel."
        )
    print(f"{server_name} MCP server is listening on http://127.0.0.1:{port}/mcp")

globals()["server_threads"] = server_threads

INFO:     Started server process [11888]
INFO:     Waiting for application startup.
StreamableHTTP session manager started
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)


Country MCP server is listening on http://127.0.0.1:8001/mcp


INFO:     Started server process [11888]
INFO:     Waiting for application startup.
StreamableHTTP session manager started
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8002 (Press CTRL+C to quit)


Weather MCP server is listening on http://127.0.0.1:8002/mcp


INFO:     Started server process [11888]
INFO:     Waiting for application startup.
StreamableHTTP session manager started
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8003 (Press CTRL+C to quit)


Flight MCP server is listening on http://127.0.0.1:8003/mcp


## 7. Connect LangChain to all three MCP servers

The adapter uses the `http` transport setting to connect to each server's Streamable HTTP endpoint.

In [9]:
client = MultiServerMCPClient(
    {
        "country": {
            "transport": "http",
            "url": "http://127.0.0.1:8001/mcp",
        },
        "weather": {
            "transport": "http",
            "url": "http://127.0.0.1:8002/mcp",
        },
        "flight": {
            "transport": "http",
            "url": "http://127.0.0.1:8003/mcp",
        },
    },
    tool_name_prefix=True,
)

tools = await client.get_tools()

print("Total tools discovered:", len(tools))
for tool in tools:
    print(f"Tool: {tool.name}")
    print("Description:", tool.description)
    print("-" * 60)

if len(tools) != 6:
    raise RuntimeError(f"Expected 6 tools from the three MCP servers, but discovered {len(tools)}.")

Created new transport with session ID: 8a57e15a1bdf4e7dba874a301591d82a
Created new transport with session ID: 0a5494a3a444494b835ed559fe367790
Created new transport with session ID: e145d32cd92c4a71ad16b5a1abfe8036


INFO:     127.0.0.1:61759 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:61760 - "POST /mcp HTTP/1.1" 200 OK


HTTP Request: POST http://127.0.0.1:8002/mcp "HTTP/1.1 200 OK"
Received session ID: 8a57e15a1bdf4e7dba874a301591d82a


INFO:     127.0.0.1:61758 - "POST /mcp HTTP/1.1" 200 OK


Negotiated protocol version: 2025-11-25
HTTP Request: POST http://127.0.0.1:8003/mcp "HTTP/1.1 200 OK"
Received session ID: 0a5494a3a444494b835ed559fe367790
Negotiated protocol version: 2025-11-25
HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
Received session ID: e145d32cd92c4a71ad16b5a1abfe8036
Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:61761 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:61762 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:61763 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:61764 - "GET /mcp HTTP/1.1" 200 OK


... (10 outputs left out)

Terminating session: 0a5494a3a444494b835ed559fe367790


INFO:     127.0.0.1:61772 - "DELETE /mcp HTTP/1.1" 200 OK


HTTP Request: DELETE http://127.0.0.1:8002/mcp "HTTP/1.1 200 OK"
GET stream disconnected, reconnecting in 1000ms...
HTTP Request: DELETE http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
HTTP Request: DELETE http://127.0.0.1:8003/mcp "HTTP/1.1 200 OK"
GET stream disconnected, reconnecting in 1000ms...
GET stream disconnected, reconnecting in 1000ms...


Total tools discovered: 6
Tool: country_get_country_details
Description: Get basic details such as capital, region, population, currencies and languages for a country.
------------------------------------------------------------
Tool: country_get_country_capital
Description: Find the capital city of a country by its name.
------------------------------------------------------------
Tool: weather_get_current_weather
Description: Get current weather conditions for a city.
------------------------------------------------------------
Tool: weather_get_weather_forecast
Description: Get a seven-day daily weather forecast for a city.
------------------------------------------------------------
Tool: flight_search_flights
Description: Search the demonstration flight dataset using airport codes such as HYD and DEL.
------------------------------------------------------------
Tool: flight_get_flight_status
Description: Get a sample flight status. This is demonstration data, not a live status fee

## 8. Test each MCP server tool before using the LLM

This separates API/server problems from Groq/agent problems.

In [10]:
def find_tool(fragment: str):
    matches = [tool for tool in tools if fragment.lower() in tool.name.lower()]
    if not matches:
        raise ValueError(
            f"No tool name contains {fragment!r}. Available tools: {[t.name for t in tools]}"
        )
    return matches[0]

country_tool = find_tool("get_country_details")
weather_tool = find_tool("get_current_weather")
flight_tool = find_tool("search_flights")

country_result = await country_tool.ainvoke({"country": "India"})
print("COUNTRY RESULT:")
print(country_result)

weather_result = await weather_tool.ainvoke({"city": "Hyderabad"})
print("\nWEATHER RESULT:")
print(weather_result)

flight_result = await flight_tool.ainvoke({"origin": "HYD", "destination": "DEL"})
print("\nFLIGHT RESULT (DEMO DATA):")
print(flight_result)

Created new transport with session ID: 66471a821cf94577b2a8caff85f1ff2f


INFO:     127.0.0.1:54404 - "POST /mcp HTTP/1.1" 200 OK


HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
Received session ID: 66471a821cf94577b2a8caff85f1ff2f
Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:54405 - "POST /mcp HTTP/1.1" 202 Accepted


HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 202 Accepted"


INFO:     127.0.0.1:54406 - "GET /mcp HTTP/1.1" 200 OK


... (22 outputs left out)

Processing request of type ListToolsRequest
HTTP Request: POST http://127.0.0.1:8003/mcp "HTTP/1.1 200 OK"
Terminating session: f77bb147e6c245dc94cfbc45e5a485ed


INFO:     127.0.0.1:57600 - "DELETE /mcp HTTP/1.1" 200 OK


HTTP Request: DELETE http://127.0.0.1:8003/mcp "HTTP/1.1 200 OK"
GET stream disconnected, reconnecting in 1000ms...



FLIGHT RESULT (DEMO DATA):
[{'type': 'text', 'text': '{\n  "flight_number": "DEMO101",\n  "airline": "Demo Air",\n  "origin": "HYD",\n  "destination": "DEL",\n  "departure": "09:00",\n  "arrival": "11:15",\n  "price_inr": 5200,\n  "status": "On time"\n}', 'id': 'lc_885dafbf-7f6f-4430-b5f5-1237f40cd9c3'}]


## 9. Create the LangChain agent

If Groq reports an authentication error, rerun the API-key cell and confirm that the key is valid and has not been revoked.

In [17]:
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key="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"
)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant with country, weather, and flight tools. "
        "Use the tools when needed. The flight server contains demonstration data only. "
        "Never describe flight results as live availability, live prices, or confirmed bookings. "
        "If airport codes are unclear, ask the user to clarify."
    ),
)

print(f"Agent created with {len(tools)} MCP tools.")

Agent created with 6 MCP tools.


## 10. Ask one question that uses all three MCP servers

In [18]:
question = (
    "Give me basic details about India, check the current weather in Hyderabad, "
    "and search the demo flight dataset for flights from HYD to DEL. "
    "Clearly label the flight result as demonstration data."
)

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": question}]}
)

print(result["messages"][-1].content)

HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Created new transport with session ID: fff589d37e7740c4b999a94ce4435d64


INFO:     127.0.0.1:50257 - "POST /mcp HTTP/1.1" 200 OK


HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
Received session ID: fff589d37e7740c4b999a94ce4435d64
Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:50258 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:50259 - "GET /mcp HTTP/1.1" 200 OK


HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 202 Accepted"
HTTP Request: GET http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"


INFO:     127.0.0.1:50260 - "POST /mcp HTTP/1.1" 200 OK


... (124 outputs left out)

Processing request of type CallToolRequest
HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
Terminating session: 9d0e2580cccd492b9fa9a17a16f9052a


INFO:     127.0.0.1:54909 - "DELETE /mcp HTTP/1.1" 200 OK


HTTP Request: DELETE http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
GET stream disconnected, reconnecting in 1000ms...
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


**Basic details about India (as of the most recent public data)**  

| Item | Information |
|------|--------------|
| **Capital** | New Delhi |
| **Region** | South Asia (part of the Indian sub‑continent) |
| **Population** | Approximately **1.42 billion** people (2023 estimate) |
| **Currency** | Indian Rupee (INR) |
| **Official languages** | Hindi (official) and English (associate official).  The constitution recognises 21 other scheduled languages, the most widely spoken being Bengali, Telugu, Marathi, Tamil, Urdu, Gujarati, Malayalam, Kannada, Odia, Punjabi, Assamese, and others. |
| **Area** | About **3.287 million km²**, making it the 7th‑largest country by land area. |
| **Time zone** | Indian Standard Time (IST) – UTC + 5:30 |
| **Government** | Federal parliamentary democratic republic; President is the head of state, Prime Minister is the head of government. |
| **Major cities** | Mumbai, Delhi, Bangalore, Hyderabad, Chennai, Kolkata, Ahmedabad, Pune, Surat, Jaipur, etc. |



## 11. Inspect agent messages and tool calls

In [19]:
for message in result["messages"]:
    print("\n" + "=" * 80)
    print("MESSAGE TYPE:", type(message).__name__)
    print("CONTENT:", message.content)
    tool_calls = getattr(message, "tool_calls", None)
    if tool_calls:
        print("TOOL CALLS:", tool_calls)


MESSAGE TYPE: HumanMessage
CONTENT: Give me basic details about India, check the current weather in Hyderabad, and search the demo flight dataset for flights from HYD to DEL. Clearly label the flight result as demonstration data.

MESSAGE TYPE: AIMessage
CONTENT: 
TOOL CALLS: [{'name': 'country_get_country_details', 'args': {'country': 'India'}, 'id': 'fc_ab54d0e6-717d-40c4-af4c-8aad5ced4f05', 'type': 'tool_call'}]

MESSAGE TYPE: ToolMessage
CONTENT: [{'type': 'text', 'text': 'Error executing tool get_country_details: 0', 'id': 'lc_ac68d257-c677-4fcb-95ce-1cb218751256'}]

MESSAGE TYPE: AIMessage
CONTENT: 
TOOL CALLS: [{'name': 'country_get_country_details', 'args': {'country': 'India'}, 'id': 'fc_57949940-04ac-4e0b-8881-769c109645b7', 'type': 'tool_call'}]

MESSAGE TYPE: ToolMessage
CONTENT: [{'type': 'text', 'text': 'Error executing tool get_country_details: 0', 'id': 'lc_308fac73-b50b-41b1-aec1-7b56e5e208a2'}]

MESSAGE TYPE: AIMessage
CONTENT: 
TOOL CALLS: [{'name': 'country_get_cou

## 12. Try more questions

In [20]:
test_questions = [
    "What is the capital of Japan?",
    "What is the weather forecast for Bengaluru?",
    "Find demo flights from DEL to LHR.",
    "What is the current weather in Hyderabad and what is India's capital?",
]

for user_question in test_questions:
    print("\nUSER:", user_question)
    answer = await agent.ainvoke(
        {"messages": [{"role": "user", "content": user_question}]}
    )
    print("AGENT:", answer["messages"][-1].content)


USER: What is the capital of Japan?


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Created new transport with session ID: 29a045f248c141c78ce12cac9a3df7a5


INFO:     127.0.0.1:54917 - "POST /mcp HTTP/1.1" 200 OK


HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
Received session ID: 29a045f248c141c78ce12cac9a3df7a5
Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:54918 - "POST /mcp HTTP/1.1" 202 Accepted


HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 202 Accepted"


... (101 outputs left out)

Processing request of type CallToolRequest
HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
Terminating session: b86b96c7fb35495b8ebc46f161c25670


INFO:     127.0.0.1:59621 - "DELETE /mcp HTTP/1.1" 200 OK


HTTP Request: DELETE http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
GET stream disconnected, reconnecting in 1000ms...
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
Retrying request to /openai/v1/chat/completions in 3.000000 seconds
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


AGENT: **Current weather in Hyderabad (India)**  
- **Temperature:** 27.8 °C  
- **Feels like:** 26.5 °C  
- **Relative humidity:** 35 %  
- **Precipitation:** 0 mm (dry)  
- **Wind:** 10.6 m/s (about 38 km/h)  
- **Conditions:** Clear / sunny (weather code 0)  

**India’s capital**  
- **New Delhi**  

*Note: The weather data is the latest available snapshot for Hyderabad; the capital information is a well‑known fact.*


## 13. Summary and troubleshooting

| Component | Responsibility |
|---|---|
| Country MCP server | Country details and capital tools |
| Weather MCP server | Current weather and seven-day forecast |
| Flight MCP server | Sample flight search and status tools |
| Streamable HTTP | Connects notebook-hosted servers without stdio subprocesses |
| `MultiServerMCPClient` | Connects to all three MCP servers |
| `get_tools()` | Discovers the six MCP tools |
| `ChatGroq` | Provides the language model |
| `create_agent()` | Creates an agent that selects and executes tools |

### Important troubleshooting

1. **`UnsupportedOperation: fileno`** — this notebook avoids the original stdio subprocess approach.
2. **`NameError: tools/agent/result is not defined`** — an earlier cell failed. Run cells from top to bottom; do not skip failed setup cells.
3. **Groq API-key error** — rerun the key cell and enter a valid `GROQ_API_KEY`.
4. **Port already in use** — restart the kernel, then run all cells again. Ports 8001, 8002 and 8003 must be available.
5. **Country/weather API errors** — check internet access and retry the relevant tool test.
6. **Flight results** — only the three hard-coded demonstration routes exist; these are not live flights.

**Security:** Never hardcode API keys in notebook cells or commit a notebook containing secrets to GitHub. Clear notebook outputs before sharing.